In [1]:
# import necessary libraries
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import UnstructuredWordDocumentLoader
from langchain_community.vectorstores import FAISS

/mnt/e/TanishProject/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [38]:
# load environment variables
load_dotenv()

# initialize LLM 
model = init_chat_model("google_genai:gemini-2.5-flash")

In [39]:
# load the required document
file_path = "NexaCorp_Enterprise_Policy_Handbook_v5.2.docx"
loader = UnstructuredWordDocumentLoader(
            file_path,
            mode="elements"             # breaks document into structured chunks based on layout
        )
docs = loader.load()

In [40]:
structured_docs = []                                    # list of processed chunks
current_section = "General"                             # default section heading

for doc in docs:
    text = doc.page_content.strip()                     # cleaned content of doc without spaces
    category = doc.metadata.get("category", "")         # type of element based on metadata

    if not text:                                        # to avoid blank chunks
        continue

    if category == "Title":                             # if element type is title, change current_section heading to text
        current_section = text
        continue

    structured_docs.append({                            # append to stuctured docs list 
        "section": current_section,
        "content": text
    })

In [46]:
merged_docs = []                                        
buffer = ""                                             # temporarily stores combined text
current_section = structured_docs[0]["section"]

for item in structured_docs:
    if item["section"] != current_section:
        if buffer:
            merged_docs.append({
                "section": current_section,
                "content": buffer
            })
        buffer = item["content"]
        current_section = item["section"]
    else:
        buffer += " " + item["content"]

if buffer:
    merged_docs.append({
        "section": current_section,
        "content": buffer
    })

print(f"Merged into {len(merged_docs)} section blocks")

Merged into 114 section blocks


In [47]:
# split the document into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,      
    chunk_overlap=80,
    separators=["\n\n", "\n", "."]  
)

final_chunks = []

for item in merged_docs:
    section = item["section"]
    text = item["content"]

    chunks = splitter.split_text(text)

    for chunk in chunks:
        if len(chunk.split()) < 25:
            continue

        final_chunks.append({
            "content": chunk,
            "section": section
        })

print(f"Final chunks created: {len(final_chunks)}")

Final chunks created: 240


In [48]:
from langchain_core.documents import Document

documents = []

for chunk in final_chunks:
    documents.append(
        Document(
            page_content=chunk["content"],
            metadata={
                "section": chunk["section"]
            }
        )
    )

print(f"Documents ready: {len(documents)}")

Documents ready: 240


In [36]:
# create embeddings
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

def embed_texts(texts):
    return embeddings.encode(texts, show_progress_bar=True)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1627.56it/s]
